In [2]:
print('test')

test


In [3]:
pip install -U langchain-text-splitters

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached orjson-3.11.9-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 6.1 MB/s  0:00:00
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached jsonpointer-3.1.1-py3-none-any.whl (7.7 kB)
Using cached orjson-3.11.9-cp312-cp312-win_amd64.whl (127 kB)
Using cached requests_toolbelt-1.0.0-py2.

In [8]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

def extract_and_split_pdf(pdf_path, heading_font_size=14.0):
    doc = fitz.open(pdf_path)
    document_segments = []
    current_heading = "Introduction"
    current_content = []

    for page in doc:
        blocks = page.get_text("blocks")
        # blocks format: (x0, y0, x1, y1, text, block_no, block_type)
        
        for b in blocks:
            text = b[4].strip()
            # Approximate font size check from the block metrics
            # Note: PyMuPDF block extraction doesn't yield direct font sizes; 
            # consider using page.get_text("dict") for exact font sizes if needed.
            if len(text) < 150 and "\n" not in text: 
                # Heuristic: Short, single-line blocks are likely headings
                if current_content:
                    document_segments.append({
                        "heading": current_heading,
                        "content": "\n".join(current_content)
                    })
                    current_content = []
                current_heading = text
            else:
                current_content.append(text)

    # Add the last section
    if current_content:
        document_segments.append({
            "heading": current_heading,
            "content": "\n".join(current_content)
        })

    # Initialize LangChain splitter for overly long sections
    ai_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )

    final_chunks = []
    for segment in document_segments:
        # Split the content if it exceeds the 1000-character limit
        splits = ai_splitter.split_text(segment["content"])
        for split in splits:
            final_chunks.append({
                "context_heading": segment["heading"],
                "chunk_text": split
            })
            print('Context Heading: ' + segment["heading"])

    return final_chunks

# Usage
# chunks = extract_and_split_pdf("your_document.pdf")


In [9]:
chunks = extract_and_split_pdf("Agents_AI.pdf")
print(chunks)

Context Heading: Full length article
Context Heading: A R T I C L E  I N F O
Context Heading: A B S T R A C T
Context Heading: A B S T R A C T
Context Heading: Prior to the widespread adoption of AI Agents and Agentic AI
Context Heading: These early studies established that individual social actions and
Context Heading: Classical Agent-like systems were designed to perform specific tasks
Context Heading: Information Fusion 126 (2026) 103599
Context Heading: R. Sapkota et al.
Context Heading: In addition to task-specific reasoning, these agents supported lim-
Context Heading: Multi-agent systems facilitated coordination among distributed en-
Context Heading: However, across these diverse systems, early AI agents shared com-
Context Heading: Recent public, academic and industry interest in AI Agents and
Context Heading: Recent public, academic and industry interest in AI Agents and
Context Heading: The release of ChatGPT in November 2022 marked a pivotal in-
Context Heading: Although the

In [10]:
chunks = extract_and_split_pdf("NIPS-2017-attention-is-all-you-need-Paper.pdf")
print(chunks)

Context Heading: Attention Is All You Need
Context Heading: Abstract
Context Heading: Abstract
Context Heading: Abstract
Context Heading: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Context Heading: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Context Heading: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Context Heading: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Context Heading: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Context Heading: Figure 1: The Transformer - model architecture.
Context Heading: Figure 1: The Transformer - model architecture.
Context Heading: 3
Context Heading: Attention(Q, K, V ) = softmax(QKT
Context Heading: Attention(Q, K, V ) = softmax(QKT
Context Heading: MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
Context Heading: The

In [12]:
import re
import fitz  # pip install pymupdf

HEADING_PATTERNS = [
    r'^\d+\.?\s+[A-Z][A-Za-z\s]{2,50}$',
    r'^[A-Z][A-Z\s]{3,40}$',
    r'^[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,5}$',
    r'^\d+\.\d+\s+[A-Z][A-Za-z\s]{2,40}$',
]

def is_heading(line: str) -> bool:
    trimmed = line.strip()
    if not trimmed or len(trimmed) >= 80:
        return False
    if len(trimmed.split()) >= 10:
        return False
    return any(re.match(p, trimmed) for p in HEADING_PATTERNS)


def trim_to_token_limit(text: str, max_tokens: int = 3000) -> str:
    max_chars = max_tokens * 4
    if len(text) <= max_chars:
        return text
    trimmed = text[:max_chars]
    last_period = trimmed.rfind('.')
    return trimmed[:last_period + 1] if last_period > 0 else trimmed


def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return '\n'.join(pages)


def split_by_headings(raw_text: str, max_tokens: int = 3000) -> list[dict]:
    lines = raw_text.split('\n')
    chunks = []
    current_heading = 'Introduction'
    current_lines = []

    for line in lines:
        trimmed = line.strip()
        if not trimmed:
            continue

        if is_heading(trimmed):
            if current_lines:
                chunks.append({
                    'heading': current_heading,
                    'text': trim_to_token_limit('\n'.join(current_lines).strip(), max_tokens),
                })
            current_heading = trimmed
            current_lines = []
        else:
            current_lines.append(trimmed)

    if current_lines:
        chunks.append({
            'heading': current_heading,
            'text': trim_to_token_limit('\n'.join(current_lines).strip(), max_tokens),
        })

    return chunks


def chunk_pdf(pdf_path: str, max_tokens: int = 3000) -> list[dict]:
    raw_text = extract_text_from_pdf(pdf_path)
    return split_by_headings(raw_text, max_tokens)

In [17]:
chunks = chunk_pdf('AttentionPaper.pdf')
print("Total chunks: " + str(len(chunks)))

for chunk in chunks:
    print(f"[{chunk['heading']}] — {len(chunk['text'])} chars")

Total chunks: 31
[Attention Is All You Need] — 15 chars
[Google Brain] — 33 chars
[Google Brain] — 28 chars
[Google Research] — 33 chars
[Google Research] — 27 chars
[Google Research] — 91 chars
[Google Brain] — 69 chars
[Abstract] — 977 chars
[Introduction] — 2979 chars
[Background] — 1815 chars
[Model Architecture] — 2096 chars
[Attention] — 5901 chars
[Positional Encoding] — 588 chars
[Layer Type] — 20 chars
[Operations] — 34 chars
[Recurrent] — 19 chars
[Convolutional] — 4370 chars
[Training] — 1109 chars
[Optimizer] — 450 chars
[Regularization] — 56 chars
[Residual Dropout] — 486 chars
[BLEU] — 486 chars
[Label Smoothing] — 171 chars
[Results] — 3 chars
[Machine Translation] — 1670 chars
[Model Variations] — 1073 chars
[Pdrop] — 13 chars
[BLEU] — 1020 chars
[Conclusion] — 1084 chars
[Acknowledgements] — 113 chars
[References] — 5404 chars
